In [37]:
import pandas as pd 
import json
import re 

In [38]:
TYPE_PATTERNS = [
    ("SRO", re.compile(r"^SR-[A-Z0-9]+(?:-[A-Z0-9]+)+$", re.IGNORECASE)),
    ("NEW_STYLE_AGENCY", re.compile(r"^S7-\d{4}-\d{2}$", re.IGNORECASE)),
    ("OLD_STYLE_AGENCY", re.compile(r"^S7-\d{2}-\d{2}$", re.IGNORECASE)),
]  

def classify_file_number(file_no: str) -> str:
    file_no = file_no.strip().upper()
    for label, pattern in TYPE_PATTERNS:
          if pattern.match(file_no):
              return label
    return "UNKNOWN_FORMAT"

In [28]:
data_file = '../data/bulk_downloads/sec/sec_2023/mapper.csv'

In [29]:
df = pd.read_csv(data_file)

In [30]:
df['url_info'] = df['url_info'].apply(json.loads)
df['sec_file_id'] = df['sec_file_id'].apply(json.loads)

In [31]:
pd.DataFrame(df['url_info'].explode().tolist())['method'].value_counts()

method
SRO_INDEX_SEARCH    6754
OLD_STYLE_AGENCY     279
UNKNOWN_FORMAT         1
Name: count, dtype: int64

In [44]:
full_sec_df = (
    df.explode(['sec_file_id', 'url_info'])
         .reset_index(drop=True)
         .pipe(lambda df: pd.concat([
             df.drop('url_info', axis=1),
             pd.DataFrame(df['url_info'].tolist())
         ], axis=1))
)

In [48]:
full_sec_df['sec_file_id'].apply(classify_file_number).value_counts()

sec_file_id
SRO                 6754
OLD_STYLE_AGENCY     276
UNKNOWN_FORMAT         4
Name: count, dtype: int64

In [55]:
full_sec_df['sec_file_id'].drop_duplicates().loc[lambda s: s.str.startswith('S7')] #.head(50)

147     S7-13-19
336     S7-10-04
337     S7-02-10
357     S7-18-15
509     S7-13-98
          ...   
7022    S7-31-08
7023    S7-18-21
7025    S7-08-09
7029    S7-21-22
7031    S7-14-22
Name: sec_file_id, Length: 90, dtype: object

In [61]:
valid_urls = full_sec_df.loc[lambda df: df['sec_file_id'].str.startswith('S7')]['primary_url']

In [59]:
import requests

In [65]:
requests.get(valid_urls.iloc[0])

<Response [403]>

In [66]:
import asyncio
from playwright.async_api import async_playwright

In [73]:
async def start_browser():
    p = await async_playwright().start()
    browser = await p.chromium.launch(headless=False)
    context = await browser.new_context()
    page = await context.new_page()
    return p, browser, context, page

p, browser, context, page = await start_browser()

In [74]:
await page.goto("https://www.example.com")

<Response url='https://www.example.com/' request=<Request url='https://www.example.com/' method='GET'>>

In [75]:
await page.goto(valid_urls.iloc[0])

<Response url='https://www.sec.gov/comments/s7-13-19/s71319.htm' request=<Request url='https://www.sec.gov/comments/s7-13-19/s71319.htm' method='GET'>>

In [77]:
html = await page.content()

In [82]:
soup = BeautifulSoup(html)

In [79]:
pwd

'/Users/spangher/Projects/stanford-research/rfi-research/regulations-demo/notebooks'

In [58]:
temp = """
<table class="usa-table views-table views-view-table cols-3">
    <caption>
              
                </caption>
          <thead>
        <tr>
                                                            <th id="view-field-comment-date-table-column" aria-sort="descending" class="views-field views-field-field-comment-date is-active" scope="col"><a href="?field_commenter_value=&amp;field_comment_letter_type_value=All&amp;field_comment_date_value%5Bmin%5D=&amp;field_comment_date_value%5Bmax%5D=&amp;order=field_comment_date&amp;sort=asc" title="sort by Date" rel="nofollow">Date<span class="tablesort tablesort--desc">
  <span class="visually-hidden">
    Sort descending
      </span>
</span>
</a></th>
                                                            <th id="view-field-comment-letter-type-1-table-column" class="views-field views-field-field-comment-letter-type-1" scope="col">Letter Type</th>
                                                            <th id="view-nothing-table-column" class="views-field views-field-nothing" scope="col">Commenter Name</th>
                  </tr>
      </thead>
        <tbody>
              <tr>
                                                                                                        <td headers="view-field-comment-date-table-column" class="views-field views-field-field-comment-date is-active">    <time datetime="2019-10-28T12:00:00Z" class="datetime">Oct. 28, 2019</time>

            </td>
                                                                                                        <td headers="view-field-comment-letter-type-1-table-column" class="views-field views-field-field-comment-letter-type-1">Public Comment            </td>
                                                                                                        <td headers="view-nothing-table-column" class="views-field views-field-nothing"><a href="/comments/s7-13-19/s71319-6355358-196251.pdf">Christopher Bok, Director, Financial Information Forum (FIF)</a>
            </td>
                  </tr>
              <tr>
                                                                                                        <td headers="view-field-comment-date-table-column" class="views-field views-field-field-comment-date is-active">    <time datetime="2019-10-28T12:00:00Z" class="datetime">Oct. 28, 2019</time>

            </td>
                                                                                                        <td headers="view-field-comment-letter-type-1-table-column" class="views-field views-field-field-comment-letter-type-1">Public Comment            </td>
                                                                                                        <td headers="view-nothing-table-column" class="views-field views-field-nothing"><a href="/comments/s7-13-19/s71319-6381876-197754.pdf">Christopher Iacovella, Chief Executive Officer, American Securities Association</a>
            </td>
                  </tr>
              <tr>
                                                                                                        <td headers="view-field-comment-date-table-column" class="views-field views-field-field-comment-date is-active">    <time datetime="2019-10-28T12:00:00Z" class="datetime">Oct. 28, 2019</time>

            </td>
                                                                                                        <td headers="view-field-comment-letter-type-1-table-column" class="views-field views-field-field-comment-letter-type-1">Public Comment            </td>
                                                                                                        <td headers="view-nothing-table-column" class="views-field views-field-nothing"><a href="/comments/s7-13-19/s71319-6355349-196250.pdf">Dennis M. Kelleher, President &amp; CEO, and Lev Bagramian, Senior Securities Policy Advisor, Better Markets, Inc.</a>
            </td>
                  </tr>
              <tr>
                                                                                                        <td headers="view-field-comment-date-table-column" class="views-field views-field-field-comment-date is-active">    <time datetime="2019-10-28T12:00:00Z" class="datetime">Oct. 28, 2019</time>

            </td>
                                                                                                        <td headers="view-field-comment-letter-type-1-table-column" class="views-field views-field-field-comment-letter-type-1">Meeting with SEC Officials            </td>
                                                                                                        <td headers="view-nothing-table-column" class="views-field views-field-nothing"><a href="/comments/s7-13-19/s71319-6399855-198367.pdf">Memorandum from the Office of the Chairman regarding an October 28, 2019, meeting with representatives of the Managed Funds Association</a>
            </td>
                  </tr>
              <tr>
                                                                                                        <td headers="view-field-comment-date-table-column" class="views-field views-field-field-comment-date is-active">    <time datetime="2019-10-28T12:00:00Z" class="datetime">Oct. 28, 2019</time>

            </td>
                                                                                                        <td headers="view-field-comment-letter-type-1-table-column" class="views-field views-field-field-comment-letter-type-1">Public Comment            </td>
                                                                                                        <td headers="view-nothing-table-column" class="views-field views-field-nothing"><a href="/comments/s7-13-19/s71319-6357609-196389.pdf">Michael Simon, Chair, CAT NMS Plan Operating Committee</a>
            </td>
                  </tr>
              <tr>
                                                                                                        <td headers="view-field-comment-date-table-column" class="views-field views-field-field-comment-date is-active">    <time datetime="2019-10-28T12:00:00Z" class="datetime">Oct. 28, 2019</time>

            </td>
                                                                                                        <td headers="view-field-comment-letter-type-1-table-column" class="views-field views-field-field-comment-letter-type-1">Public Comment            </td>
                                                                                                        <td headers="view-nothing-table-column" class="views-field views-field-nothing"><a href="/comments/s7-13-19/s71319-6366765-195937.pdf">Theodore R. Lazo, Managing Director and Associate General Counsel, and Ellen Greene, Managing Director, Financial Services Operations; SIFMA</a>
            </td>
                  </tr>
              <tr>
                                                                                                        <td headers="view-field-comment-date-table-column" class="views-field views-field-field-comment-date is-active">    <time datetime="2019-10-28T12:00:00Z" class="datetime">Oct. 28, 2019</time>

            </td>
                                                                                                        <td headers="view-field-comment-letter-type-1-table-column" class="views-field views-field-field-comment-letter-type-1">Public Comment            </td>
                                                                                                        <td headers="view-nothing-table-column" class="views-field views-field-nothing"><a href="/comments/s7-13-19/s71319-6357608-196387.pdf">Thomas Tesauro, President, Fidelity Capital Markets</a>
            </td>
                  </tr>
              <tr>
                                                                                                        <td headers="view-field-comment-date-table-column" class="views-field views-field-field-comment-date is-active">    <time datetime="2019-10-03T12:00:00Z" class="datetime">Oct. 3, 2019</time>

            </td>
                                                                                                        <td headers="view-field-comment-letter-type-1-table-column" class="views-field views-field-field-comment-letter-type-1">Meeting with SEC Officials            </td>
                                                                                                        <td headers="view-nothing-table-column" class="views-field views-field-nothing"><a href="/comments/s7-13-19/s71319-6252407-192807.pdf">Memorandum from the Office of Commissioner Allison Herren Lee regarding an October 3, 2019, meeting with representatives of Nasdaq</a>
            </td>
                  </tr>
              <tr>
                                                                                                        <td headers="view-field-comment-date-table-column" class="views-field views-field-field-comment-date is-active">    <time datetime="2018-05-01T12:00:00Z" class="datetime">May 1, 2018</time>

            </td>
                                                                                                        <td headers="view-field-comment-letter-type-1-table-column" class="views-field views-field-field-comment-letter-type-1">Public Comment            </td>
                                                                                                        <td headers="view-nothing-table-column" class="views-field views-field-nothing"><a href="/comments/s7-13-19/s71319-6090109-191884.pdf">Brett Redfearn, Director, Division of Trading and Markets, U.S. Securities and Exchange Commission</a>
            </td>
                  </tr>
          </tbody>
  </table>

"""

In [81]:
from bs4 import BeautifulSoup